# 第 14 课｜数据集、批量训练与验证集

**把第 13 课“能拟合几句话”的程序，升级为能够处理变长文本、批量训练和留出评估的实验。**

本课仍用 `Embedding → Linear` 的一字符条件模型。改进的是**数据流程与评估方法**，不是模型的上下文能力；它还不是 Transformer。

你将完成：

`文本文件 → 固定规则清理 → 按整句切分 → 训练集词表 → Dataset → 动态补齐 → DataLoader → 训练/验证 → 保存最佳权重`

CPU 即可，无需显卡、下载语料或模型。Notebook 与配套 `.py` 均独立完整，不依赖第 13 课文件，也不互相导入。

**学习方式：** 先完成第 1–7 节的数据与 padding 实验，再完成第 8–13 节的训练评估，最后独立做三道练习。大段的测试和保存辅助函数可在第二遍阅读。

## 0. 环境与文件约定

沿用已经能运行 PyTorch 的环境。未安装时可在当前 Notebook 新单元执行 `%pip install torch`，然后重启内核；已经安装则不要重复操作。显卡版安装参照文末官方入口 [7]。

本讲义的实际验证环境会在下方打印；不要求你重装成相同版本。默认 `DEVICE=cpu`。已有 CUDA 环境可改为 `choose_device("auto")`。

默认输入 `lesson_14_data/corpus.txt` 位于**当前工作目录**下：缺失才创建，已有则保留。输出到 `lesson_14_outputs`，同名生成文件会覆盖。Notebook 从不同文件夹启动，文件位置也可能不同，以下打印绝对路径供你确认。

不要在不了解单元依赖时跳着运行；修改前面的定义后，需要重新运行后续使用它的单元。

In [1]:
from __future__ import annotations
import argparse
from functools import partial
import json
import math
from pathlib import Path
import random
import tempfile
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

SPECIAL_TOKENS = ["<unk>", "<bos>", "<eos>", "<pad>"]
IGNORE_INDEX = -100

def choose_device(requested: str = "cpu") -> torch.device:
    if requested not in {"cpu", "cuda", "auto"}:
        raise ValueError("device 必须是 cpu、cuda 或 auto")
    if requested == "auto":
        requested = "cuda" if torch.cuda.is_available() else "cpu"
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA 不可用；请先用 --device cpu，或检查本机驱动和 PyTorch")
    return torch.device(requested)

# 只对这类微型 CPU 实验设置单线程；不是通用的性能配置建议。
torch.set_num_threads(1)
DEVICE = choose_device("cpu")
SEED = 14
DATA_PATH = Path("lesson_14_data/corpus.txt")
OUTPUT_DIR = Path("lesson_14_outputs")
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available(), "本课设备:", DEVICE)
print("工作目录:", Path.cwd())
print("输入文件:", DATA_PATH.resolve())

PyTorch: 2.10.0+cpu
CUDA 可用: False 本课设备: cpu
工作目录: /mnt/data/_lesson14_work/notebook_run
输入文件: /mnt/data/_lesson14_work/notebook_run/lesson_14_data/corpus.txt


## 1. 文本文件：一行一条样本

第 13 课只有四句等长文本。本课用六个主语与六个短语组合出 36 句，再增加 4 句短文本，得到 40 条不同文本。为观察数据清理，输入额外包含 2 条重复与 2 条空白行。

本课明确选择以下固定规则：去掉两端空白、删除空行、按首次出现顺序去掉完全相同的整句；内部空格保留。**这是教学数据约定，不是所有任务都应采用的清理方法。** 代码、诗歌等任务的空白可能有意义。

`set` 用来记住已出现文本，`list` 保持顺序。下面先创建或读取文件，再打印清理统计。后面切分前就去掉完全重复的句子，避免相同文本被分到两边。

In [2]:
def demo_lines() -> list[str]:
    """40 条不同的手工模板句，再加入两条重复和两条空白用于演示清理。"""
    subjects = ["我", "你", "他", "我们", "你们", "他们"]
    endings = ["爱学习。", "爱模型。", "喜欢学习。", "喜欢写代码。", "每天读书。", "正在学习。"]
    lines = [subject + ending for subject in subjects for ending in endings]
    lines += ["学习。", "读书。", "写代码。", "模型。"]
    return lines + [lines[0], "  " + lines[1] + "  ", "", "   "]


def ensure_demo_file(path: Path) -> Path:
    """缺失才创建。以独占创建模式避免覆盖已有输入。"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with path.open("x", encoding="utf-8") as handle:
            handle.write("\n".join(demo_lines()) + "\n")
    except FileExistsError:
        pass
    return path


def clean_lines(lines: list[str]) -> list[str]:
    """固定规则：去两端空白、删除空行、保留首次出现的完整文本。内部空格保留。"""
    if not isinstance(lines, list) or any(not isinstance(line, str) for line in lines):
        raise ValueError("lines 必须是字符串列表")
    seen = set()
    cleaned = []
    for line in lines:
        text = line.strip()
        if text and text not in seen:
            cleaned.append(text)
            seen.add(text)
    return cleaned


def read_corpus(path: Path) -> tuple[list[str], dict]:
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    texts = clean_lines(lines)
    nonblank_count = sum(bool(line.strip()) for line in lines)
    stats = {
        "raw_lines": len(lines),
        "blank_removed": len(lines) - nonblank_count,
        "duplicates_removed": nonblank_count - len(texts),
        "unique_texts": len(texts),
    }
    return texts, stats

ensure_demo_file(DATA_PATH)
texts, cleaning_stats = read_corpus(DATA_PATH)
print("清理统计:", cleaning_stats)
print("前 6 条:", texts[:6])
print("字符长度范围:", min(map(len, texts)), "到", max(map(len, texts)))

清理统计: {'raw_lines': 44, 'blank_removed': 2, 'duplicates_removed': 2, 'unique_texts': 40}
前 6 条: ['我爱学习。', '我爱模型。', '我喜欢学习。', '我喜欢写代码。', '我每天读书。', '我正在学习。']
字符长度范围: 3 到 8


## 2. 训练集和验证集：先分开，再学习数据规则

**训练集**用于反向传播和更新参数；**验证集**用于观察留出表现、选择轮次。本课没有第三份独立测试集。

处理顺序：固定的文本清理与去重 → 按整条文本切分 → **只用训练侧建立词表** → 两侧共用同一词表。需要从数据中学习的预处理规则，不应在切分前使用所有样本拟合 [6]。这里不使用 scikit-learn 包，只参考这个原则。

默认种子 `14`、验证比例 `0.25`：40 条文本分成 30 条训练、10 条验证。`random.Random(seed)` 是独立随机源；相同输入与种子产生相同切分。

**边界：** 全句不重复不等于没有所有泄漏。我们的两侧共享模板和词语，所以这只是同一套教学模板里的留出实验。真实任务常需按来源文档、用户、时间等分组，并检查近重复；不能先切滑动窗口、再把同文档的重叠片段随机散到两边。

In [3]:
def split_texts(texts: list[str], val_fraction: float = 0.25, seed: int = 14) -> tuple[list[str], list[str]]:
    """先按整条文本切分，后建词表；不把同一句的 token 随机分到两边。"""
    if not isinstance(texts, list) or len(texts) < 4:
        raise ValueError("清理后至少需要 4 条不同的非空文本")
    if any(not isinstance(t, str) or not t.strip() for t in texts):
        raise ValueError("切分输入必须是非空字符串列表")
    if len(set(texts)) != len(texts):
        raise ValueError("切分前请先去掉完整文本重复项")
    if isinstance(val_fraction, bool) or not isinstance(val_fraction, (int, float)):
        raise ValueError("val_fraction 必须是 0 到 1 之间的数")
    if not math.isfinite(val_fraction) or not 0 < val_fraction < 1:
        raise ValueError("val_fraction 必须严格在 0 和 1 之间")
    if type(seed) is not int:
        raise ValueError("seed 必须是整数")
    shuffled = list(texts)
    random.Random(seed).shuffle(shuffled)  # 局部随机源，不修改 Python 全局随机状态。
    n_val = max(1, min(len(texts) - 1, int(len(texts) * val_fraction)))
    return shuffled[n_val:], shuffled[:n_val]

train_texts, val_texts = split_texts(texts, val_fraction=0.25, seed=SEED)
print("训练条数:", len(train_texts), "验证条数:", len(val_texts))
print("整句重合:", set(train_texts) & set(val_texts))
print("训练示例:", train_texts[:3])
print("验证示例:", val_texts[:3])

训练条数: 30 验证条数: 10
整句重合: set()
训练示例: ['我喜欢学习。', '学习。', '他爱模型。']
验证示例: ['读书。', '你们爱模型。', '我们每天读书。']


## 3. 增加 `<pad>`，并处理验证集的新字符

| 标记 | 本课编号 | 作用 |
|---|---:|---|
| `<unk>` | 0 | 训练词表未收录的字符 |
| `<bos>` | 1 | 句子开始 |
| `<eos>` | 2 | 句子结束，是需要学习的目标 |
| `<pad>` | 3 | 为组成规则张量而补齐的输入位置 |

普通字符从 4 开始。与第 13 课相比新增了 PAD，**普通字符编号不保证相同，不能直接混用上一课的权重与词表**。

下面仅用 `train_texts` 建词表，再编码不在默认语料中的“量子”。不认识的字变成 `<unk>`，不会偷偷扩大词表。默认数据的验证集不一定有未知字，所以我们单独构造这个边界实验。

未知字符率按原文字符数计算；损失按 token 计算并包含 EOS，两者分母不同。若大量字符变成同一个 UNK，指标已在衡量压缩后的任务，不能当作对原字符的准确理解。

In [4]:
def build_vocab(train_texts: list[str]) -> tuple[dict[str, int], list[str]]:
    """只用训练文本建词表；编号顺序与首次出现顺序一致。"""
    if not isinstance(train_texts, list) or not train_texts:
        raise ValueError("train_texts 必须是非空字符串列表")
    id_to_token = list(SPECIAL_TOKENS)
    token_to_id = {token: i for i, token in enumerate(id_to_token)}
    for text in train_texts:
        if not isinstance(text, str):
            raise ValueError("每条训练文本必须是字符串")
        for char in text:
            if char not in token_to_id:
                token_to_id[char] = len(id_to_token)
                id_to_token.append(char)
    return token_to_id, id_to_token


def validate_vocab(token_to_id: dict[str, int]) -> None:
    if not isinstance(token_to_id, dict) or any(t not in token_to_id for t in SPECIAL_TOKENS):
        raise ValueError("词表必须包含 <unk>、<bos>、<eos>、<pad>")
    ids = list(token_to_id.values())
    if any(type(i) is not int for i in ids) or sorted(ids) != list(range(len(ids))):
        raise ValueError("词表编号必须唯一、连续，且从 0 开始")


def encode(text: str, token_to_id: dict[str, int]) -> list[int]:
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")
    return [token_to_id.get(char, token_to_id["<unk>"]) for char in text]


def unknown_char_rate(texts: list[str], token_to_id: dict[str, int]) -> dict:
    total = sum(len(text) for text in texts)
    unknown = sum(char not in token_to_id for text in texts for char in text)
    return {"unknown_chars": unknown, "chars": total, "rate": unknown / total if total else 0.0}

token_to_id, id_to_token = build_vocab(train_texts)
PAD_ID = token_to_id["<pad>"]
print("词表大小:", len(id_to_token))
print("有序词表:", id_to_token)
print("量子 的编号:", encode("量子", token_to_id))
print("验证集未知字符统计:", unknown_char_rate(val_texts, token_to_id))

词表大小: 25
有序词表: ['<unk>', '<bos>', '<eos>', '<pad>', '我', '喜', '欢', '学', '习', '。', '他', '爱', '模', '型', '你', '们', '正', '在', '写', '代', '码', '每', '天', '读', '书']
量子 的编号: [0, 0]
验证集未知字符统计: {'unknown_chars': 0, 'chars': 56, 'rate': 0.0}


## 4. Dataset：负责取出一条样本

`Dataset` 保存样本，并实现两个方法：`__len__()` 回答“有多少条”，`__getitem__(index)` 回答“第 index 条是什么”。`DataLoader` 之后负责分批遍历它 [1]。

每条文本先变为完整序列，再移位一次：

```text
原文：我
完整：[BOS, 我, EOS]
x：  [BOS, 我]
y：  [我, EOS]
```

一个普通字符数为 `L` 的句子，有 `L+1` 个有效预测目标，其中最后一个是 EOS。本课不在损失函数中再次移位。

文件读取阶段删除空行；但 `CharDataset([""])` 本身仍支持空文本，得到 `BOS → EOS`。这是“数据清理政策”和“函数边界行为”的区别。

In [5]:
class CharDataset(Dataset):
    """一条文本是一条样本；先加 BOS/EOS，再移位一次；数据留在 CPU。"""

    def __init__(self, texts: list[str], token_to_id: dict[str, int]) -> None:
        validate_vocab(token_to_id)
        if not isinstance(texts, list) or not texts:
            raise ValueError("数据集需要非空的文本列表")
        self.sequences = []
        for text in texts:
            ids = [token_to_id["<bos>"]] + encode(text, token_to_id) + [token_to_id["<eos>"]]
            self.sequences.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self) -> int:
        return len(self.sequences)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        full_ids = self.sequences[index]
        return full_ids[:-1], full_ids[1:]

train_ds = CharDataset(train_texts, token_to_id)
val_ds = CharDataset(val_texts, token_to_id)
short_ds = CharDataset(["我", "我爱学习。", ""], token_to_id)
for index in range(len(short_ds)):
    x, y = short_ds[index]
    print(index, "x:", [id_to_token[i] for i in x.tolist()],
          "y:", [id_to_token[i] for i in y.tolist()])

0 x: ['<bos>', '我'] y: ['我', '<eos>']
1 x: ['<bos>', '我', '爱', '学', '习', '。'] y: ['我', '爱', '学', '习', '。', '<eos>']
2 x: ['<bos>'] y: ['<eos>']


## 5. 动态 padding：长度不同，怎么放进同一个 batch？

“我”的预测长度为 2，“我爱学习。”的预测长度为 6。同批时先把较短的补到 6：

| 句子 | 输入 x | 目标 y |
|---|---|---|
| 我 | BOS 我 PAD PAD PAD PAD | 我 EOS -100 -100 -100 -100 |
| 我爱学习。 | BOS 我 爱 学 习 。 | 我 爱 学 习 。 EOS |

**输入用词表内的 PAD 编号，标签用忽略值 `-100`。** `-100` 不是 token，不可送进 Embedding。

`pad_sequence(..., batch_first=True)` 生成 `[B,T]`，其中 `T` 是**本批**最长长度，不是整个数据集最长长度 [2]。不同批次的 `T` 可以不同。

`collate_fn` 就是“如何把多条样本整理成一批”的函数。除了 x、y，本课也返回每条样本的有效长度，便于检查。

In [6]:
def collate_batch(samples: list[tuple[torch.Tensor, torch.Tensor]], *, pad_id: int) -> dict[str, torch.Tensor]:
    """本批最长长度动态补齐：输入补 PAD，标签补 -100；有效 EOS 不忽略。"""
    if not isinstance(samples, list) or not samples:
        raise ValueError("不能整理空批次")
    if type(pad_id) is not int or pad_id < 0:
        raise ValueError("pad_id 必须是非负整数")
    xs, ys, lengths = [], [], []
    for pair in samples:
        if not isinstance(pair, (tuple, list)) or len(pair) != 2:
            raise ValueError("每条样本必须包含 input_ids 和 targets")
        x, y = pair
        if not all(isinstance(t, torch.Tensor) and t.ndim == 1 and t.dtype == torch.long for t in pair):
            raise ValueError("样本的 x/y 必须是一维 long 张量")
        if x.device.type != "cpu" or y.device.type != "cpu":
            raise ValueError("本课先在 CPU 整理批次，再整体移动到计算设备")
        if x.numel() == 0 or x.shape != y.shape:
            raise ValueError("样本 x/y 必须非空且长度一致")
        if (x < 0).any().item() or (y < 0).any().item():
            raise ValueError("原始样本不能含负编号；-100 仅由补齐步骤写入标签")
        xs.append(x)
        ys.append(y)
        lengths.append(x.numel())
    return {
        "input_ids": pad_sequence(xs, batch_first=True, padding_value=pad_id),
        "targets": pad_sequence(ys, batch_first=True, padding_value=IGNORE_INDEX),
        "lengths": torch.tensor(lengths, dtype=torch.long),
    }

small_batch = collate_batch([short_ds[i] for i in range(len(short_ds))], pad_id=PAD_ID)
print("input_ids:\n", small_batch["input_ids"])
print("targets:\n", small_batch["targets"])
print("lengths:", small_batch["lengths"].tolist())
print("形状:", tuple(small_batch["input_ids"].shape))
print("有效目标数:", int((small_batch["targets"] != IGNORE_INDEX).sum()))

input_ids:
 tensor([[ 1,  4,  3,  3,  3,  3],
        [ 1,  4, 11,  7,  8,  9],
        [ 1,  3,  3,  3,  3,  3]])
targets:
 tensor([[   4,    2, -100, -100, -100, -100],
        [   4,   11,    7,    8,    9,    2],
        [   2, -100, -100, -100, -100, -100]])
lengths: [2, 6, 1]
形状: (3, 6)
有效目标数: 9


## 6. DataLoader：分批、打乱顺序、保留最后小批次

本课训练时 `shuffle=True`，打乱的是**整条样本的遍历顺序**，不是句子内部的字符。验证用 `shuffle=False`。`drop_last=False` 表示末尾不满一批也保留。

30 条训练样本、`batch_size=8`，每个 epoch 有 4 个 batch，大小依次为 8、8、8、6。一个 epoch 是完整遍历训练集一次；本课每个 batch 更新一次，60 个 epoch 共 240 次更新。

`partial(collate_batch, pad_id=PAD_ID)` 可以理解成“先固定 pad_id 的整理函数”。`num_workers=0` 先在主进程取数据，降低 Notebook 与 Windows 下的入门配置复杂度；后续再学习并行加载 [1]。

In [7]:
def make_loader(dataset: CharDataset, *, batch_size: int, pad_id: int, shuffle: bool, seed: int = 14) -> DataLoader:
    if type(batch_size) is not int or batch_size < 1:
        raise ValueError("batch_size 必须是正整数")
    # partial 固定 pad_id；DataLoader 调用时只需再提供 samples。
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        collate_fn=partial(collate_batch, pad_id=pad_id),
        num_workers=0, drop_last=False,
        generator=torch.Generator().manual_seed(seed),
    )

train_loader = make_loader(train_ds, batch_size=8, pad_id=PAD_ID, shuffle=True, seed=SEED)
train_eval_loader = make_loader(train_ds, batch_size=8, pad_id=PAD_ID, shuffle=False, seed=SEED)
val_loader = make_loader(val_ds, batch_size=8, pad_id=PAD_ID, shuffle=False, seed=SEED)
for batch_index, batch in enumerate(train_loader):
    valid_count = int((batch["targets"] != IGNORE_INDEX).sum())
    all_slots = batch["targets"].numel()
    print("batch", batch_index, "shape", tuple(batch["input_ids"].shape),
          "有效 token", valid_count, "补齐位置", all_slots - valid_count)

batch 0 shape (8, 9) 有效 token 55 补齐位置 17
batch 1 shape (8, 9) 有效 token 58 补齐位置 14
batch 2 shape (8, 8) 有效 token 58 补齐位置 6
batch 3 shape (6, 9) 有效 token 44 补齐位置 10


## 7. 模型没有变复杂：依旧是逐位置预测

结构仍为 `[B,T] → Embedding → [B,T,D] → Linear → [B,T,V]`。

本课把 `padding_idx=PAD_ID` 传给 Embedding：默认初始化时该行置零，普通查表反传不会给该行贡献梯度 [4]。但这不是损失忽略设置，也不是注意力掩码。

| 机制 | 解决的问题 | 本课是否实现 |
|---|---|---|
| 输入 padding | 组成规则形状的张量 | 是 |
| loss mask | 哪些目标位置参与损失与指标 | 是 |
| attention mask | 哪些位置允许互相读取信息 | 本模型没有注意力，不在此实现 |

未来换成 Transformer，需要另外处理因果约束与补齐位置的信息交互。**仅仅忽略 padding 标签，并不能保证注意力不读取 padding。**

In [8]:
class TinyNextTokenLM(nn.Module):
    """沿用第 13 课的逐位置模型。padding_idx 不等于 loss mask 或 attention mask。"""

    def __init__(self, vocab_size: int, embedding_dim: int = 16, pad_id: int = 3) -> None:
        super().__init__()
        if type(vocab_size) is not int or vocab_size < 4:
            raise ValueError("vocab_size 至少为 4")
        if type(embedding_dim) is not int or embedding_dim < 1:
            raise ValueError("embedding_dim 必须是正整数")
        if type(pad_id) is not int or not 0 <= pad_id < vocab_size:
            raise ValueError("pad_id 必须在词表范围内")
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.lm_head = nn.Linear(embedding_dim, vocab_size)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        if not isinstance(input_ids, torch.Tensor) or input_ids.ndim != 2 or input_ids.numel() == 0:
            raise ValueError("input_ids 必须是非空 [B,T] 张量")
        if input_ids.dtype != torch.long:
            raise ValueError("Embedding 输入必须是 long 编号")
        if input_ids.min().item() < 0 or input_ids.max().item() >= self.vocab_size:
            raise ValueError("Embedding 输入编号越界；不要把标签 -100 作为输入")
        return self.lm_head(self.embedding(input_ids))

torch.manual_seed(SEED)
demo_model = TinyNextTokenLM(len(id_to_token), embedding_dim=16, pad_id=PAD_ID)
demo_logits = demo_model(small_batch["input_ids"])
print("logits 形状:", tuple(demo_logits.shape))
print("PAD embedding:", demo_model.embedding.weight[PAD_ID].detach())
try:
    demo_model(small_batch["targets"])
except ValueError as error:
    print("故意把标签当输入会被拒绝:", error)

logits 形状: (3, 6, 25)
PAD embedding: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
故意把标签当输入会被拒绝: Embedding 输入编号越界；不要把标签 -100 作为输入


## 8. 忽略补齐目标：EOS 仍然要学习

对整数标签，交叉熵的 `ignore_index=-100` 让相应**目标位置**不贡献损失 [3]。

我们先得到有效位置损失总和，再除以数量：

$$L_{batch}=\frac{\sum_{b,t:y_{b,t}\ne -100}\ell_{b,t}}{\sum_{b,t}\mathbf{1}[y_{b,t}\ne-100]}.$$

此处没有类别权重，也没有 label smoothing。EOS、UNK 都是正常目标；只有 -100 被忽略。全是 -100 的批次没有有效分母，本课主动报错，而不是制造一个看似正常的数。

```python
loss_sum = F.cross_entropy(
    logits.reshape(-1, vocab_size), targets.reshape(-1),
    ignore_index=-100, reduction="sum",
)
loss = loss_sum / valid_count
```

`masked_loss_sum()` 返回的损失仍是可微张量；不要在 `backward()` 前把它变成 Python 浮点数。

In [9]:
def masked_loss_sum(logits: torch.Tensor, targets: torch.Tensor) -> tuple[torch.Tensor, int]:
    """返回有效位置的损失之和和数量；保持可反传，不提前 .item()。"""
    if not isinstance(logits, torch.Tensor) or logits.ndim != 3 or logits.numel() == 0:
        raise ValueError("logits 必须是非空 [B,T,V] 张量")
    if not isinstance(targets, torch.Tensor) or targets.shape != logits.shape[:2]:
        raise ValueError("targets 必须是对应的 [B,T] 张量")
    if not logits.is_floating_point() or targets.dtype != torch.long or logits.device != targets.device:
        raise ValueError("logits/targets 类型或设备不匹配")
    if not torch.isfinite(logits).all().item():
        raise ValueError("logits 必须全部有限")
    valid = targets != IGNORE_INDEX
    count = int(valid.sum().item())
    if count == 0:
        raise ValueError("这个批次没有有效目标，不能计算平均损失")
    valid_ids = targets[valid]
    if valid_ids.min().item() < 0 or valid_ids.max().item() >= logits.shape[-1]:
        raise ValueError("有效标签编号越界")
    loss_sum = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]), targets.reshape(-1),
        ignore_index=IGNORE_INDEX, reduction="sum",
    )
    if not torch.isfinite(loss_sum).item():
        raise ValueError("损失溢出；检查数据、学习率和计算精度")
    return loss_sum, count


def masked_accuracy_counts(logits: torch.Tensor, targets: torch.Tensor) -> tuple[int, int]:
    valid = targets != IGNORE_INDEX
    correct = ((logits.argmax(dim=-1) == targets) & valid).sum().item()
    return int(correct), int(valid.sum().item())

demo_targets = small_batch["targets"]
loss_sum, valid_count = masked_loss_sum(demo_logits, demo_targets)
reference = F.cross_entropy(demo_logits[demo_targets != -100], demo_targets[demo_targets != -100])
print("有效目标:", valid_count)
print("正确平均损失:", (loss_sum / valid_count).item())
print("只取有效位置计算:", reference.item())

# 错误演示：把标签 -100 换成 PAD 并纳入平均，就把补齐当成了预测任务。
wrong_targets = demo_targets.masked_fill(demo_targets == -100, PAD_ID)
wrong_loss = F.cross_entropy(demo_logits.reshape(-1, len(id_to_token)), wrong_targets.reshape(-1))
print("错误地训练 PAD 的平均损失:", wrong_loss.item())

有效目标: 9
正确平均损失: 3.4806439876556396
只取有效位置计算: 3.4806439876556396
错误地训练 PAD 的平均损失: 3.3496110439300537


### 实验：增加补齐位置，正确损失应保持不变

我们保持所有有效 logits 不变，只追加两个无效位置。如果损失或有效目标计数改变了，说明忽略逻辑或分母有问题。

还要区分“忽略目标位置”和“删除输出类别”：词表中依然有 PAD 类，它在**有效位置**的 softmax 分母中仍存在。`ignore_index` 并没有从每个位置的输出向量里删掉 PAD 这一列 [3]。

In [10]:
extra_logits = torch.full((demo_logits.shape[0], 2, demo_logits.shape[-1]), 500.0)
extra_targets = torch.full((demo_targets.shape[0], 2), -100, dtype=torch.long)
padded_logits = torch.cat([demo_logits, extra_logits], dim=1)
padded_targets = torch.cat([demo_targets, extra_targets], dim=1)
padded_sum, padded_count = masked_loss_sum(padded_logits, padded_targets)
print("追加前后有效目标:", valid_count, padded_count)
print("追加前后损失一致:", torch.allclose(loss_sum, padded_sum))

probe = demo_logits.detach().clone().requires_grad_(True)
probe_loss, _ = masked_loss_sum(probe, demo_targets)
probe_loss.backward()
print("被忽略位置的梯度绝对值之和:", probe.grad[demo_targets == -100].abs().sum().item())
print("有效位置上 PAD 类仍有梯度:", probe.grad[..., PAD_ID][demo_targets != -100].abs().sum().item() > 0)

追加前后有效目标: 9 9
追加前后损失一致: True
被忽略位置的梯度绝对值之和: 0.0
有效位置上 PAD 类仍有梯度: True


## 9. 汇总验证损失：不要直接平均 batch 的均值

设第一批有 9 个有效 token，每个损失都是 1；第二批只有 1 个有效 token，损失为 9。

错误的平均：`(1 + 9) / 2 = 5`。

本课采用的 token 平均：`(9 × 1 + 1 × 9) / 10 = 1.8`。

较长句子贡献更多 token，这是我们明确选定的度量。若任务需要“每句等权”，需要另外定义和报告该指标，不能混为一谈。

验证集总体公式：

$$L_{val}=\frac{\sum_{batches}\mathrm{loss\_sum}}{\sum_{batches}\mathrm{valid\_count}},\qquad
\mathrm{accuracy}=\frac{\mathrm{有效位置预测正确总数}}{\mathrm{有效位置总数}}.$$

使用自然对数交叉熵时，本课定义困惑度 `PPL = exp(L_val)`。它依赖分词、词表、数据和包含哪些目标，不能随意跨设置比较。这里默认包含 EOS 与 UNK，不包含 padding。

In [11]:
def metrics_from_totals(nll_sum: float, correct: int, tokens: int) -> dict:
    """按 token 汇总，不平均各批的 mean loss；自然对数下 PPL=exp(loss)。"""
    if tokens <= 0 or not math.isfinite(nll_sum) or nll_sum < 0 or not 0 <= correct <= tokens:
        raise ValueError("指标总量不合法或没有有效 token")
    loss = nll_sum / tokens
    try:
        perplexity = math.exp(loss)
    except OverflowError:
        perplexity = None  # JSON 中写 null，避免把溢出伪装成正常数值。
    return {"loss": loss, "perplexity": perplexity, "token_accuracy": correct / tokens, "tokens": tokens}

batch_means = [1.0, 9.0]
batch_counts = [9, 1]
wrong_mean = sum(batch_means) / len(batch_means)
loss_total = sum(mean * count for mean, count in zip(batch_means, batch_counts))
right_mean = loss_total / sum(batch_counts)
print("直接平均批均值:", wrong_mean)
print("按有效 token 加权:", right_mean)
print("汇总函数:", metrics_from_totals(loss_total, correct=7, tokens=10))

直接平均批均值: 5.0
按有效 token 加权: 1.8
汇总函数: {'loss': 1.8, 'perplexity': 6.0496474644129465, 'token_accuracy': 0.7, 'tokens': 10}


## 10. 验证函数：固定参数、关闭求导、恢复原模式

`model.eval()` 设置模块的评估行为；`torch.no_grad()` 禁止记录这段计算的反向图。两者不是一回事 [5]。本模型没有 Dropout/BatchNorm，`eval()` 不改变其数学输出，但保留这个规范方便未来扩展。

验证中不应调用 `loss.backward()` 或 `optimizer.step()`。已存在的 `.grad` 也不必清空：`no_grad()` 不会清除先前的梯度。我们用 `try/finally` 确保结束后恢复调用前模式。

下面用同一模型与同一验证数据，分别采用 batch size 1、3、8 评估。**结果应在浮点误差范围内一致**。这个测试适用于本课逐位置模型；对未来模型，还需确保注意力、位置编号和掩码同样正确。

In [12]:
def evaluate(model: TinyNextTokenLM, loader: DataLoader, device: torch.device) -> dict:
    """固定当前参数完整评估；不反传、不 step，恢复调用前模式。"""
    was_training = model.training
    model.eval()
    total_loss, total_correct, total_tokens = 0.0, 0, 0
    try:
        with torch.no_grad():
            for batch in loader:
                x = batch["input_ids"].to(device)
                y = batch["targets"].to(device)
                logits = model(x)
                loss_sum, count = masked_loss_sum(logits, y)
                correct, _ = masked_accuracy_counts(logits, y)
                total_loss += loss_sum.item()
                total_correct += correct
                total_tokens += count
    finally:
        model.train(was_training)
    return metrics_from_totals(total_loss, total_correct, total_tokens)


def clone_state(model: nn.Module) -> dict[str, torch.Tensor]:
    """真正复制最佳权重；不能只保留随后会继续变化的 state_dict 引用。"""
    return {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}

cpu = torch.device("cpu")
state_before = clone_state(demo_model)
for size in [1, 3, 8]:
    eval_loader = make_loader(val_ds, batch_size=size, pad_id=PAD_ID, shuffle=False)
    result = evaluate(demo_model, eval_loader, cpu)
    print("batch_size", size, "loss", round(result["loss"], 7), "有效 token", result["tokens"])
print("验证前后参数相同:", all(torch.equal(state_before[k], v) for k, v in demo_model.state_dict().items()))
print("恢复训练模式:", demo_model.training)

batch_size 1 loss 3.3215629 有效 token 66
batch_size 3 loss 3.3215628 有效 token 66
batch_size 8 loss 3.321563 有效 token 66
验证前后参数相同: True
恢复训练模式: True


## 11. 每批训练，每轮评估

每批训练：清零梯度 → 前向计算 → 有效 token 平均损失 → 反向传播 → 更新参数。数据先在 CPU 整理，再按整批 `.to(device)`。

注意两个常见混淆：

**在线训练损失（online）**：一轮中累积各批更新前的损失，批次之间模型在变化。

**轮末训练评估损失**：固定这一轮结束时的参数，再完整评估一次训练集。

比较训练与验证时，本课使用后者与验证损失配对。每轮还按验证损失选择最佳权重，连第 0 轮的初始模型也作为候选，避免训练变差时丢掉更好的起点。

最佳参数必须复制保存；只写 `best_state = model.state_dict()` 可能仍指向之后继续变化的张量。本课使用 `detach().cpu().clone()`。

In [13]:
def train_one_epoch(model: TinyNextTokenLM, loader: DataLoader, optimizer: torch.optim.Optimizer, device: torch.device) -> dict:
    """一个 epoch 遍历训练集一次，每批更新一次；不保证每次更新 loss 都下降。"""
    model.train()
    total_loss, total_correct, total_tokens = 0.0, 0, 0
    for batch in loader:
        x = batch["input_ids"].to(device)
        y = batch["targets"].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss_sum, count = masked_loss_sum(logits, y)
        loss = loss_sum / count  # 当前 batch 的 token 平均梯度。
        loss.backward()
        optimizer.step()
        correct, _ = masked_accuracy_counts(logits.detach(), y)
        total_loss += loss_sum.detach().item()
        total_correct += correct
        total_tokens += count
    # 这些是训练途中的 online 指标，参数在批间变化，不是固定模型的完整评估。
    return metrics_from_totals(total_loss, total_correct, total_tokens)


def fit_model(model: TinyNextTokenLM, train_loader: DataLoader, train_eval_loader: DataLoader,
              val_loader: DataLoader, device: torch.device, *, epochs: int = 60,
              lr: float = 0.03, verbose: bool = True) -> dict:
    if type(epochs) is not int or epochs < 1:
        raise ValueError("epochs 必须是正整数")
    if isinstance(lr, bool) or not isinstance(lr, (int, float)) or not math.isfinite(lr) or lr <= 0:
        raise ValueError("lr 必须是正的有限数")
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    initial_train = evaluate(model, train_eval_loader, device)
    initial_val = evaluate(model, val_loader, device)
    history = [{"epoch": 0, "train": initial_train, "validation": initial_val}]
    best_loss, best_epoch, best_state = initial_val["loss"], 0, clone_state(model)
    for epoch in range(1, epochs + 1):
        online = train_one_epoch(model, train_loader, optimizer, device)
        train_metrics = evaluate(model, train_eval_loader, device)
        val_metrics = evaluate(model, val_loader, device)
        history.append({"epoch": epoch, "online_train": online, "train": train_metrics, "validation": val_metrics})
        if val_metrics["loss"] < best_loss:
            best_loss, best_epoch, best_state = val_metrics["loss"], epoch, clone_state(model)
        if verbose and (epoch == 1 or epoch % 10 == 0 or epoch == epochs):
            print(f"epoch {epoch:3d} | train loss {train_metrics['loss']:.4f} | val loss {val_metrics['loss']:.4f}")
    final_metrics = history[-1]
    model.load_state_dict(best_state)
    selected_train = evaluate(model, train_eval_loader, device)
    selected_val = evaluate(model, val_loader, device)
    return {
        "history": history, "best_epoch": best_epoch,
        "initial_train": initial_train, "initial_validation": initial_val,
        "last_epoch": final_metrics,
        "selected_train": selected_train, "selected_validation": selected_val,
    }

## 12. 完整实验与保存：最佳轮次不一定是最后一轮

下面的辅助函数把前面步骤连起来，并保存：

| 文件 | 内容 |
|---|---|
| `split.json` | 实际训练/验证文本与切分种子 |
| `vocab.json` | 有顺序的词表 |
| `metrics.json` | 配置、清理统计、未知字符率、各轮指标及重载核对 |
| `best_model.pt` | 所选模型权重、结构配置、词表与轮次 |

这不是精确续训检查点：未保存优化器状态、随机数状态和数据读取位置。只加载自己生成或信任的文件。

默认训练 60 轮，每轮 4 次更新。以下 `run_experiment` 会重新建立 loader 并设置种子，因此前面观察 batch 的单元不会消耗正式训练的采样顺序。固定种子有助于同环境复现，不承诺跨 PyTorch 版本、设备得到逐位一致结果。

In [14]:
def save_artifacts(output_dir: Path, model: TinyNextTokenLM, token_to_id: dict[str, int],
                   id_to_token: list[str], train_texts: list[str], val_texts: list[str],
                   report: dict) -> Path:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    split = {"seed": report["config"]["seed"], "train": train_texts, "validation": val_texts}
    for name, data in [("split.json", split), ("vocab.json", id_to_token), ("metrics.json", report)]:
        (output_dir / name).write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    checkpoint = {
        "model_state": clone_state(model),
        "model_config": {"vocab_size": model.vocab_size, "embedding_dim": model.embedding_dim, "pad_id": model.pad_id},
        "id_to_token": list(id_to_token), "best_epoch": report["training"]["best_epoch"],
    }
    path = output_dir / "best_model.pt"
    torch.save(checkpoint, path)
    return path


def load_checkpoint(path: Path, device: torch.device) -> tuple[TinyNextTokenLM, dict[str, int], list[str]]:
    """只加载自己生成或信任的文件；不是精确续训检查点，没有优化器/RNG 状态。"""
    checkpoint = torch.load(Path(path), map_location="cpu", weights_only=True)
    id_to_token = checkpoint["id_to_token"]
    token_to_id = {token: i for i, token in enumerate(id_to_token)}
    validate_vocab(token_to_id)
    model = TinyNextTokenLM(**checkpoint["model_config"]).to(device)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    return model, token_to_id, id_to_token


def run_experiment(text_file: Path, output_dir: Path, *, device: torch.device,
                   epochs: int = 60, batch_size: int = 8, seed: int = 14,
                   val_fraction: float = 0.25, embedding_dim: int = 16,
                   lr: float = 0.03, verbose: bool = True) -> dict:
    text_file, output_dir = Path(text_file), Path(output_dir)
    generated_names = ["split.json", "vocab.json", "metrics.json", "best_model.pt"]
    if any(text_file.resolve() == (output_dir / name).resolve() for name in generated_names):
        raise ValueError("输入文件不能与即将生成的结果文件重名")
    texts, cleaning = read_corpus(text_file)
    train_texts, val_texts = split_texts(texts, val_fraction, seed)
    token_to_id, id_to_token = build_vocab(train_texts)  # 切分后，只读取训练侧。
    train_ds = CharDataset(train_texts, token_to_id)
    val_ds = CharDataset(val_texts, token_to_id)
    pad_id = token_to_id["<pad>"]
    train_loader = make_loader(train_ds, batch_size=batch_size, pad_id=pad_id, shuffle=True, seed=seed)
    train_eval_loader = make_loader(train_ds, batch_size=batch_size, pad_id=pad_id, shuffle=False, seed=seed)
    val_loader = make_loader(val_ds, batch_size=batch_size, pad_id=pad_id, shuffle=False, seed=seed)
    torch.manual_seed(seed)
    model = TinyNextTokenLM(len(id_to_token), embedding_dim, pad_id).to(device)
    if verbose:
        print(f"训练 {len(train_ds)} 条 | 验证 {len(val_ds)} 条 | 词表 {len(id_to_token)} | 设备 {device}")
    training = fit_model(model, train_loader, train_eval_loader, val_loader, device,
                         epochs=epochs, lr=lr, verbose=verbose)
    report = {
        "lesson": 14, "torch_version": str(torch.__version__), "device": str(device),
        "input_file": str(text_file.resolve()), "cleaning": cleaning,
        "config": {"epochs": epochs, "batch_size": batch_size, "seed": seed,
                   "val_fraction": val_fraction, "embedding_dim": embedding_dim, "lr": lr},
        "data": {"train_texts": len(train_ds), "validation_texts": len(val_ds),
                 "train_batches": len(train_loader), "validation_batches": len(val_loader),
                 "vocab_size": len(id_to_token),
                 "train_oov": unknown_char_rate(train_texts, token_to_id),
                 "validation_oov": unknown_char_rate(val_texts, token_to_id)},
        "parameter_count": sum(p.numel() for p in model.parameters()), "training": training,
        "caveats": ["教学模板语料，整句不重合不等于模板独立", "验证集用于选择轮次，不是独立测试集",
                    "一字符条件模型，没有新增上下文能力", "损失统计包含 EOS/UNK，不包含 -100 补齐位置"],
    }
    checkpoint_path = save_artifacts(output_dir, model, token_to_id, id_to_token, train_texts, val_texts, report)
    restored, restored_vocab, _ = load_checkpoint(checkpoint_path, device)
    if restored_vocab != token_to_id:
        raise AssertionError("重新加载的词表顺序发生变化")
    restored_val = evaluate(restored, val_loader, device)
    if not math.isclose(restored_val["loss"], training["selected_validation"]["loss"], rel_tol=1e-6, abs_tol=1e-6):
        raise AssertionError("重新加载后验证损失不一致")
    report["reload_validation"] = restored_val
    (output_dir / "metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    if verbose:
        print(f"所选轮次：{training['best_epoch']}；验证 loss：{restored_val['loss']:.6f}")
        print(f"结果目录：{output_dir.resolve()}")
    return report

In [15]:
report = run_experiment(
    DATA_PATH, OUTPUT_DIR, device=DEVICE,
    epochs=60, batch_size=8, seed=SEED,
    val_fraction=0.25, embedding_dim=16, lr=0.03,
)
training_report = report["training"]
print("参数量:", report["parameter_count"])
for label, metrics in [
    ("初始训练", training_report["initial_train"]),
    ("初始验证", training_report["initial_validation"]),
    ("所选训练", training_report["selected_train"]),
    ("所选验证", training_report["selected_validation"]),
    ("末轮训练", training_report["last_epoch"]["train"]),
    ("末轮验证", training_report["last_epoch"]["validation"]),
]:
    print(label, "loss", round(metrics["loss"], 6), "accuracy", round(metrics["token_accuracy"], 4))

训练 30 条 | 验证 10 条 | 词表 25 | 设备 cpu


epoch   1 | train loss 2.0511 | val loss 2.2241
epoch  10 | train loss 0.5057 | val loss 0.8001
epoch  20 | train loss 0.5007 | val loss 0.8349
epoch  30 | train loss 0.5020 | val loss 0.8546
epoch  40 | train loss 0.5010 | val loss 0.8564


epoch  50 | train loss 0.4982 | val loss 0.8821
epoch  60 | train loss 0.4973 | val loss 0.8700
所选轮次：7；验证 loss：0.774760
结果目录：/mnt/data/_lesson14_work/notebook_run/lesson_14_outputs
参数量: 825
初始训练 loss 3.277429 accuracy 0.0
初始验证 loss 3.321563 accuracy 0.0
所选训练 loss 0.521764 accuracy 0.7535
所选验证 loss 0.77476 accuracy 0.7424
末轮训练 loss 0.497327 accuracy 0.7581
末轮验证 loss 0.870044 accuracy 0.7273


### 如何解释结果，而不是只看一个下降的数字？

在本次已保存的 CPU 运行中，默认语料的训练 30 条、验证 10 条，词表 25，参数量 825。第 7 轮验证损失最低：约 `0.774760`。训练到第 60 轮，训练损失更低，验证损失却约为 `0.870044`。

这是**本次有限样本上的观察**，不意味着所有实验都在第 7 轮最好，也不能只凭一轮的波动断言过拟合。请结合曲线趋势、样本量、优化过程与数据分布判断。

验证集已被用来选最佳轮次，因此“最佳验证成绩”不是独立最终测试成绩。还需要未参与选择的测试集才能更独立地评估；本课不把这 40 条模板句包装成模型能力评测。

另一个误区：`batch_size` 改变后重新训练，不保证最终参数相同，因为更新次数、梯度构成和优化轨迹改变了。**固定模型的评估分批不变性**，不等于**不同 batch size 训练的等价性**。

In [16]:
restored_model, restored_vocab, restored_tokens = load_checkpoint(OUTPUT_DIR / "best_model.pt", DEVICE)
restored_val_ds = CharDataset(val_texts, restored_vocab)
for size in [1, 4, 10]:
    loader = make_loader(restored_val_ds, batch_size=size, pad_id=restored_vocab["<pad>"], shuffle=False)
    metrics = evaluate(restored_model, loader, DEVICE)
    print("重载后 batch size", size, "验证 loss", round(metrics["loss"], 7), "token", metrics["tokens"])
print("保存文件:", sorted(p.name for p in OUTPUT_DIR.iterdir() if p.is_file()))

重载后 batch size 1 验证 loss 0.7747603 token 66
重载后 batch size 4 验证 loss 0.7747603 token 66
重载后 batch size 10 验证 loss 0.7747603 token 66
保存文件: ['best_model.pt', 'metrics.json', 'split.json', 'vocab.json']


## 13. 换成自己的文本文件

先备份已有实验。准备 UTF-8 文本，一行一个独立样本，清理后至少 4 条不同非空文本。尽量先从几十条短文本开始；本课不做截断或长度桶，极长行会让整个 batch 补得很长。

终端命令（替换路径为你自己的文件）：

```bash
python lesson_14_datasets_batches_and_validation.py --text-file my_text.txt --output-dir my_run_14
```

脚本只会自动创建默认示例输入；显式指定的文件不存在时会报错，不会替你生成一个同名文件。输出目录中同名生成文件会覆盖。失败后磁盘上可能留有旧结果，应先看退出状态与错误提示。

自己的文本中存在空格时，内部空格保留。输入字符串 `"<pad>"` 会按普通字符编码，并不会自动变为控制 token；特殊 token 是程序直接插入对应编号。

不要只追求更低训练 loss。先检查切分是否独立、未知率是否过高、标签是否正确、补齐位置是否被忽略，再解释数值。

---
## 14. 独立练习：先在这里停下来

下面函数默认未实现，检查开关为 `False`。填好后改为 `True` 再运行。参考答案位于后面，使用不同函数名，不覆盖你的实现。

**练习 A：整理变长批次。** 实现 `collate_ex(samples, pad_id)`。输入是合法的一维 long x/y 样本对；返回 `input_ids`、`targets`、`lengths` 字典。按本批最长长度补齐，标签填 -100。拒绝空 samples 与 x/y 长度不匹配，不要再移位标签。

**练习 B：计算掩码平均损失。** 实现 `token_mean_loss_ex(logits, targets)`，返回可反传的标量。忽略 -100；有效分母为 0 时主动报错。不要先 softmax，也不要提前 `.item()`。

**练习 C：独立验证函数。** 实现 `evaluate_ex(model, loader, device)`，返回 `loss`、`token_accuracy`、`tokens`。允许复用本课的损失辅助函数，但不能调用已有 `evaluate()` 代替实现。要求关闭梯度、恢复模式、按有效 token 汇总，且没有参数更新。

In [17]:
def collate_ex(samples, pad_id):
    # TODO：取出 x/y，记录有效长度，分别补齐。
    raise NotImplementedError("请完成练习 A")


def token_mean_loss_ex(logits, targets):
    # TODO：有效目标计数，计算损失之和，再除以正确的数量。
    raise NotImplementedError("请完成练习 B")


def evaluate_ex(model, loader, device):
    # TODO：保存模式，eval + no_grad，累积总量，finally 恢复模式。
    raise NotImplementedError("请完成练习 C")

In [18]:
import math
import torch
from torch import nn
from torch.nn import functional as F


def check_exercise_a(fn):
    samples = [
        (torch.tensor([1, 4]), torch.tensor([4, 2])),
        (torch.tensor([1, 4, 5, 6]), torch.tensor([4, 5, 6, 2])),
    ]
    result = fn(samples, 3)
    assert result["input_ids"].tolist() == [[1, 4, 3, 3], [1, 4, 5, 6]]
    assert result["targets"].tolist() == [[4, 2, -100, -100], [4, 5, 6, 2]]
    assert result["lengths"].tolist() == [2, 4]
    assert result["input_ids"].dtype == torch.long and result["targets"].dtype == torch.long
    for invalid in [[], [(torch.tensor([1, 4]), torch.tensor([2]))]]:
        try:
            fn(invalid, 3)
        except ValueError:
            pass
        else:
            raise AssertionError("应该拒绝空批次或不匹配长度")
    print("练习 A 检查通过")


def check_exercise_b(fn):
    logits = torch.tensor([[[1., 2., 0.], [2., 0., 1.], [9., -2., 3.]]], requires_grad=True)
    targets = torch.tensor([[1, 2, -100]])
    expected = F.cross_entropy(logits[0, :2], targets[0, :2])
    actual = fn(logits, targets)
    assert isinstance(actual, torch.Tensor) and actual.ndim == 0 and actual.requires_grad
    assert torch.allclose(actual, expected)
    extended = torch.cat([logits, torch.full((1, 2, 3), 444.)], dim=1)
    extended_y = torch.cat([targets, torch.full((1, 2), -100, dtype=torch.long)], dim=1)
    assert torch.allclose(fn(extended, extended_y), expected)
    actual.backward()
    assert torch.count_nonzero(logits.grad[0, 2]).item() == 0
    assert logits.grad[0, :2].abs().sum().item() > 0
    try:
        fn(logits.detach(), torch.full_like(targets, -100))
    except ValueError:
        pass
    else:
        raise AssertionError("应该拒绝全忽略的目标")
    print("练习 B 检查通过")


def check_exercise_c(fn):
    class ProbeModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.table = nn.Parameter(torch.tensor([[1., -1., 0.], [0., 2., -1.], [-1., 0., 1.]]))
            self.grad_enabled_flags = []
        def forward(self, x):
            self.grad_enabled_flags.append(torch.is_grad_enabled())
            return self.table[x]

    model = ProbeModel()
    batches = [
        {"input_ids": torch.tensor([[0, 1], [2, 0]]), "targets": torch.tensor([[1, -100], [2, 1]])},
        {"input_ids": torch.tensor([[0]]), "targets": torch.tensor([[1]])},
    ]
    before = model.table.detach().clone()
    flat_logits = torch.stack([before[0], before[2], before[0], before[0]])
    flat_y = torch.tensor([1, 2, 1, 1])
    expected = F.cross_entropy(flat_logits, flat_y).item()
    expected_accuracy = (flat_logits.argmax(-1) == flat_y).float().mean().item()
    model.train()
    result = fn(model, batches, torch.device("cpu"))
    assert math.isclose(result["loss"], expected, rel_tol=1e-6, abs_tol=1e-6)
    assert result["tokens"] == 4
    assert result["token_accuracy"] == expected_accuracy
    assert torch.equal(model.table.detach(), before) and model.table.grad is None
    assert model.training and not any(model.grad_enabled_flags)
    model.eval()
    fn(model, batches, torch.device("cpu"))
    assert not model.training
    print("练习 C 检查通过")

In [19]:
RUN_A = False
RUN_B = False
RUN_C = False

if RUN_A:
    check_exercise_a(collate_ex)
if RUN_B:
    check_exercise_b(token_mean_loss_ex)
if RUN_C:
    check_exercise_c(evaluate_ex)
if not any([RUN_A, RUN_B, RUN_C]):
    print("练习尚未验收：请实现函数并打开对应检查开关。")

练习尚未验收：请实现函数并打开对应检查开关。


### 两个变化实验

**固定模型，改变评估分批。** 加载同一个 `best_model.pt`，分别设置验证 batch size 为 1、4、10。解释为什么损失应近似一致，而重新训练时改变 batch size 并没有这种保证。

**验证数据出现新字。** 用只包含“我爱学习。”的文本建词表，再构建“我爱量子。”的数据集，记录 UNK 数量。确认验证数据没有扩大词表。解释为什么不能用“UNK 总能预测对”推断模型认识了这些新字。

先写预测，再运行。不要为了让输出符合预期而修改参考答案或跳过边界检查。

---
## 15. 参考答案

先独立写过，再阅读。每个答案只需要标准库和 PyTorch，不调用你的练习函数，也不覆盖它们。

In [20]:
import torch
from torch.nn.utils.rnn import pad_sequence


def collate_answer(samples, pad_id):
    if not samples:
        raise ValueError("不能整理空批次")
    xs, ys, lengths = [], [], []
    for x, y in samples:
        if x.numel() == 0 or x.shape != y.shape:
            raise ValueError("x/y 必须非空且长度相同")
        xs.append(x)
        ys.append(y)
        lengths.append(x.numel())
    return {
        "input_ids": pad_sequence(xs, batch_first=True, padding_value=pad_id),
        "targets": pad_sequence(ys, batch_first=True, padding_value=-100),
        "lengths": torch.tensor(lengths, dtype=torch.long),
    }

check_exercise_a(collate_answer)

练习 A 检查通过


In [21]:
import torch
from torch.nn import functional as F


def token_mean_loss_answer(logits, targets):
    count = int((targets != -100).sum().item())
    if count == 0:
        raise ValueError("没有有效目标")
    total = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]), targets.reshape(-1),
        ignore_index=-100, reduction="sum",
    )
    return total / count

check_exercise_b(token_mean_loss_answer)

练习 B 检查通过


In [22]:
import torch
from torch.nn import functional as F


def evaluate_answer(model, loader, device):
    was_training = model.training
    total_loss, total_correct, total_tokens = 0.0, 0, 0
    model.eval()
    try:
        with torch.no_grad():
            for batch in loader:
                x = batch["input_ids"].to(device)
                y = batch["targets"].to(device)
                logits = model(x)
                valid = y != -100
                count = int(valid.sum().item())
                if count == 0:
                    raise ValueError("没有有效目标")
                loss_sum = F.cross_entropy(
                    logits.reshape(-1, logits.shape[-1]), y.reshape(-1),
                    ignore_index=-100, reduction="sum",
                )
                total_loss += loss_sum.item()
                total_correct += int(((logits.argmax(-1) == y) & valid).sum().item())
                total_tokens += count
    finally:
        model.train(was_training)
    if total_tokens == 0:
        raise ValueError("评估数据为空")
    return {
        "loss": total_loss / total_tokens,
        "token_accuracy": total_correct / total_tokens,
        "tokens": total_tokens,
    }

check_exercise_c(evaluate_answer)

练习 C 检查通过


## 16. 自动检查与脚本入口

下方固定检查与脚本共用同一份实现，验证数据切分、移位、padding、梯度、评估分批不变性、最佳权重及保存重载等行为。它们是**代码行为测试**，不是模型语言能力测验。

检查中的极小训练用例为了快速验证函数，会重复用同一小样本集合；该用例不用于报告泛化成绩。正式实验始终按整句划分训练与验证。

终端运行：

```bash
python lesson_14_datasets_batches_and_validation.py
python lesson_14_datasets_batches_and_validation.py --checks-only
python lesson_14_datasets_batches_and_validation.py --device auto
```

脚本默认先运行 CPU 固定检查，再在选择的设备训练。独立导入脚本不会自动生成文件或启动训练。CPU 路径已实测；本次运行环境没有可用 CUDA，未在 RTX 3060 上实测。

In [23]:
def run_checks() -> int:
    passed = 0

    def check(condition: bool, message: str) -> None:
        nonlocal passed
        if not bool(condition):
            raise AssertionError(message)
        passed += 1

    def rejects(fn, exception=ValueError) -> None:
        nonlocal passed
        try:
            fn()
        except exception:
            passed += 1
        else:
            raise AssertionError("应拒绝这个输入，却没有报错")

    def close(a: float, b: float, tol: float = 1e-6) -> bool:
        return math.isclose(a, b, abs_tol=tol, rel_tol=tol)

    cpu = torch.device("cpu")
    check(clean_lines([" 我 ", "我", "", "  ", "我 爱"]) == ["我", "我 爱"], "清理或去重规则错误")
    rejects(lambda: clean_lines([1]))
    raw = demo_lines()
    cleaned = clean_lines(raw)
    check(len(raw) == 44 and len(cleaned) == 40, "示例语料规模错误")
    train, val = split_texts(cleaned)
    check(len(train) == 30 and len(val) == 10, "切分比例错误")
    check(not set(train) & set(val), "整句切分有重合")
    check(set(train) | set(val) == set(cleaned), "切分有丢失")
    check(split_texts(cleaned) == (train, val), "同种子切分不一致")
    check(split_texts(cleaned, seed=15) != (train, val), "不同种子没有改变切分")
    check(cleaned == clean_lines(raw), "切分修改了原始列表")
    rejects(lambda: split_texts(["我"] * 4))
    rejects(lambda: split_texts(["我", "你", "他"]))
    rejects(lambda: split_texts(cleaned, val_fraction=1.0))
    rejects(lambda: split_texts(cleaned, val_fraction=float("nan")))
    rejects(lambda: split_texts(cleaned, seed=True))
    t2i, i2t = build_vocab(["我爱学习。", "你爱模型。"])
    check(i2t[:4] == SPECIAL_TOKENS, "特殊编号顺序错误")
    check("量" not in t2i and encode("量", t2i) == [t2i["<unk>"]], "未知字符处理错误")
    before_vocab = dict(t2i)
    oov_ds = CharDataset(["量子"], t2i)
    check(t2i == before_vocab, "构建验证数据改变了词表")
    check(oov_ds[0][1].tolist() == [0, 0, 2], "未知标签或 EOS 错误")
    rejects(lambda: validate_vocab({"<unk>": 0}))
    rejects(lambda: build_vocab([]))
    rejects(lambda: CharDataset([], t2i))
    ds = CharDataset(["我", "我爱学习。", ""], t2i)
    check(len(ds) == 3, "Dataset 长度错误")
    check(ds[0][0].tolist() == [1, t2i["我"]], "输入对齐错误")
    check(ds[0][1].tolist() == [t2i["我"], 2], "标签未移位或多移位")
    check(ds[2][0].tolist() == [1] and ds[2][1].tolist() == [2], "空文本应保留 BOS->EOS")
    check(ds[0][0].device.type == "cpu" and ds[0][0].dtype == torch.long, "Dataset 设备或 dtype 错误")
    batch = collate_batch([ds[i] for i in range(3)], pad_id=3)
    x, y = batch["input_ids"], batch["targets"]
    check(x.shape == (3, 6) and y.shape == (3, 6), "动态 padding 形状错误")
    check(batch["lengths"].tolist() == [2, 6, 1], "有效长度错误")
    check(x[0].tolist() == [1, t2i["我"], 3, 3, 3, 3], "输入 padding 错误")
    check(y[0].tolist() == [t2i["我"], 2, -100, -100, -100, -100], "标签 padding 错误")
    check(int((y != -100).sum()) == 9, "不能把 padding 计为有效 token")
    check(int((y == 2).sum()) == 3, "EOS 必须参与损失")
    rejects(lambda: collate_batch([], pad_id=3))
    rejects(lambda: collate_batch([(torch.tensor([1.0]), torch.tensor([2]))], pad_id=3))
    rejects(lambda: collate_batch([(torch.tensor([1, 2]), torch.tensor([2]))], pad_id=3))
    rejects(lambda: collate_batch([(torch.tensor([-100]), torch.tensor([2]))], pad_id=3))
    loader = make_loader(ds, batch_size=2, pad_id=3, shuffle=False)
    batches = list(loader)
    check([b["input_ids"].shape[0] for b in batches] == [2, 1], "末尾小批次被丢弃")
    check([b["input_ids"].shape[1] for b in batches] == [6, 1], "不是按本批最长长度补齐")
    rejects(lambda: make_loader(ds, batch_size=0, pad_id=3, shuffle=False))
    torch.manual_seed(14)
    model = TinyNextTokenLM(len(i2t), 8, 3)
    logits = model(x)
    check(logits.shape == (3, 6, len(i2t)), "模型输出形状错误")
    check(torch.equal(model.embedding.weight[3], torch.zeros(8)), "初始 padding 行不为零")
    rejects(lambda: model(y))
    loss_sum, n = masked_loss_sum(logits, y)
    reference = F.cross_entropy(logits[y != -100], y[y != -100], reduction="sum")
    check(n == 9 and torch.allclose(loss_sum, reference), "掩码损失与显式筛选不一致")
    padded_logits = torch.cat([logits, torch.ones(3, 2, len(i2t)) * 987.0], dim=1)
    padded_y = torch.cat([y, torch.full((3, 2), -100, dtype=torch.long)], dim=1)
    ext_sum, ext_n = masked_loss_sum(padded_logits, padded_y)
    check(ext_n == n and torch.allclose(ext_sum, loss_sum), "增加 padding 改变损失")
    probe = logits.detach().clone().requires_grad_(True)
    probe_sum, _ = masked_loss_sum(probe, y)
    probe_sum.backward()
    check(torch.count_nonzero(probe.grad[y == -100]).item() == 0, "忽略位置仍有 logits 梯度")
    check(probe.grad[y != -100].abs().sum().item() > 0, "有效位置没有梯度")
    check(probe.grad[..., 3][y != -100].abs().sum().item() > 0, "忽略位置不等于删除 PAD 输出类别")
    model.zero_grad(set_to_none=True)
    loss_sum.backward()
    check(torch.count_nonzero(model.embedding.weight.grad[3]).item() == 0, "padding embedding 有梯度")
    check(torch.isfinite(model.lm_head.weight.grad).all().item(), "输出层梯度不有限")
    rejects(lambda: masked_loss_sum(logits, torch.full_like(y, -100)))
    rejects(lambda: masked_loss_sum(logits, y.float()))
    bad_y = y.clone(); bad_y[0, 0] = len(i2t)
    rejects(lambda: masked_loss_sum(logits, bad_y))
    bad_y = y.clone(); bad_y[0, 0] = -7
    rejects(lambda: masked_loss_sum(logits, bad_y))
    rejects(lambda: masked_loss_sum(logits[:, :-1], y))
    rejects(lambda: masked_loss_sum(logits * float("nan"), y))
    totals = metrics_from_totals(18.0, 5, 10)
    check(close(totals["loss"], 1.8) and close(totals["token_accuracy"], 0.5), "token 加权汇总错误")
    check(close(totals["perplexity"], math.exp(1.8)), "PPL 公式错误")
    check(not close(totals["loss"], (1.0 + 9.0) / 2), "不能平均各批均值")
    rejects(lambda: metrics_from_totals(0.0, 0, 0))
    correct_logits = torch.zeros(3, 6, len(i2t))
    for row in range(3):
        for col in range(6):
            target = int(y[row, col])
            correct_logits[row, col, target if target != -100 else 0] = 10
    check(masked_accuracy_counts(correct_logits, y) == (9, 9), "准确率分母包含 padding")
    model.train()
    state_before = clone_state(model)
    grad_before = {name: p.grad.clone() if p.grad is not None else None for name, p in model.named_parameters()}
    eval2 = evaluate(model, loader, cpu)
    eval1 = evaluate(model, make_loader(ds, batch_size=1, pad_id=3, shuffle=False), cpu)
    eval3 = evaluate(model, make_loader(ds, batch_size=3, pad_id=3, shuffle=False), cpu)
    check(close(eval1["loss"], eval2["loss"]) and close(eval2["loss"], eval3["loss"]), "改变评估 batch size 改变了 loss")
    check(eval1["tokens"] == eval2["tokens"] == eval3["tokens"] == 9, "评估 token 数错误")
    check(eval1["token_accuracy"] == eval3["token_accuracy"], "准确率受评估分批影响")
    check(model.training, "evaluate 未恢复训练模式")
    check(all(torch.equal(state_before[k], v) for k, v in model.state_dict().items()), "验证修改了参数")
    check(all(p.grad is None if grad_before[name] is None else torch.equal(p.grad, grad_before[name]) for name, p in model.named_parameters()), "验证修改了已有梯度")
    model.eval(); evaluate(model, loader, cpu)
    check(not model.training, "evaluate 未恢复原评估模式")
    model.train()
    bad_loader = [{"input_ids": torch.tensor([[1]]), "targets": torch.tensor([[-100]])}]
    rejects(lambda: evaluate(model, bad_loader, cpu))
    check(model.training, "评估出错后模式没有恢复")
    tiny_ds = CharDataset(["我", "我我", "爱", "我爱"], t2i)
    tiny_train = make_loader(tiny_ds, batch_size=2, pad_id=3, shuffle=True)
    tiny_eval = make_loader(tiny_ds, batch_size=3, pad_id=3, shuffle=False)
    smoke = fit_model(model, tiny_train, tiny_eval, tiny_eval, cpu, epochs=12, verbose=False)
    check(smoke["selected_train"]["loss"] < smoke["initial_train"]["loss"], "教学训练未降低训练 loss")
    check(close(smoke["selected_validation"]["loss"], min(r["validation"]["loss"] for r in smoke["history"])), "最佳权重不是验证最优权重")
    check(0 <= smoke["best_epoch"] <= 12, "最佳轮次不合法")
    check(torch.equal(model.embedding.weight[3], torch.zeros(8)), "本课 Adam 无权重衰减时 PAD 行变化")
    # 上面 train=val 只用于快速检查训练函数，不对外报告泛化指标。
    with tempfile.TemporaryDirectory() as tmp:
        root = Path(tmp)
        corpus_path = ensure_demo_file(root / "corpus.txt")
        _, cleaning = read_corpus(corpus_path)
        check(cleaning == {"raw_lines": 44, "blank_removed": 2, "duplicates_removed": 2, "unique_texts": 40}, "文件读取清理统计错误")
        corpus_path.write_text("保留这段输入\n", encoding="utf-8")
        ensure_demo_file(corpus_path)
        check(corpus_path.read_text(encoding="utf-8") == "保留这段输入\n", "已有输入被覆盖")
        rejects(lambda: read_corpus(root / "missing.txt"), FileNotFoundError)
        report = {"config": {"seed": 14}, "training": smoke}
        path = save_artifacts(root / "out", model, t2i, i2t, ["我"], ["爱"], report)
        restored, rt2i, ri2t = load_checkpoint(path, cpu)
        check(rt2i == t2i and ri2t == i2t, "保存后词表顺序变化")
        with torch.no_grad():
            check(torch.equal(model(x), restored(x)), "保存重载后 logits 不一致")
        check(json.loads((root / "out" / "split.json").read_text(encoding="utf-8"))["validation"] == ["爱"], "切分记录未保存")
    print(f"通过固定检查：{passed} 项（CPU）。")
    return passed

check_count = run_checks()

通过固定检查：79 项（CPU）。


## 17. 常见问题与验收标准

| 现象 | 先检查 |
|---|---|
| `stack` 报不同长度 | 是否给 DataLoader 设置了 `collate_fn`？ |
| Embedding 出现负索引 | 是否把标签 -100 传成了输入？ |
| 添加 padding 后 loss 改变 | 是否忘了 ignore，或者仍除以 `B*T`？ |
| EOS 学不会 | 是否错误地把 EOS 当作 padding 忽略？ |
| 验证 batch size 改了，loss 明显变了 | 是否平均了各批 mean loss，或漏了最后小批？ |
| 验证运行后参数变化 | 是否在验证里调用了 backward/step，或没正确设置模式？ |
| 长文本占满内存 | 是否让单条极长样本抬高了整个 batch 的补齐长度？ |
| best 模型其实是最后一轮 | 是否只保存了 state_dict 的引用，而没有复制？ |

本课完成标准：你能从原文写出 x/y/padding 的对应关系，独立完成三道练习，解释为什么验证没有梯度更新，以及为什么最低验证 loss 不能当作独立测试成绩。

下一课：**位置表示、LayerNorm 与残差连接**。在已有注意力、MLP 和训练流程的基础上，为组装完整 Transformer block 做准备。

## 官方资料与本课范围

API 行为参考以下官方文档；教学语料、数值例子、实现和验收测试为本课构造。链接指向官方维护的文档，版本可能随时间变化。本课实际执行版本在第 0 节与输出的 `metrics.json` 中记录。

[1] [PyTorch：Datasets & DataLoaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html)  
[2] [PyTorch：pad_sequence](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html)  
[3] [PyTorch：CrossEntropyLoss，ignore_index 与 reduction](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)  
[4] [PyTorch：Embedding，padding_idx](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)  
[5] [PyTorch：Autograd mechanics，eval 与 no-grad 的区别](https://docs.pytorch.org/docs/stable/notes/autograd.html)  
[6] [scikit-learn：数据泄漏与预处理](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)  
[7] [PyTorch 官方安装选择器](https://pytorch.org/get-started/locally/)

本课未涉及：BPE、长文本滑窗、packing、多进程预取、分布式训练、真实评测基准以及 Transformer 的 padding attention mask。这里不使用复杂的数据框架，以便先看懂每一个输入、标签、梯度和统计量的来源。